# Random Forest Code Extracted from Screenshots

This notebook contains the Random Forest sampling code extracted from your uploaded screenshots.

It demonstrates:

1. Creating a synthetic classification dataset
2. Converting it into a DataFrame
3. Row sampling
4. Feature sampling
5. Combined row + feature sampling
6. Training multiple Decision Tree models
7. Visualizing a Decision Tree
8. Making predictions

This is the manual process behind Random Forest: different trees are trained on different random rows and/or features.


In [ ]:
import numpy as np
import pandas as pd
import random

from sklearn.datasets import make_classification

In [ ]:
X, y = make_classification(
    n_features=5,
    n_redundant=0,
    n_informative=5,
    n_clusters_per_class=1,
    random_state=42
)

In [ ]:
df = pd.DataFrame(
    X,
    columns=["col1", "col2", "col3", "col4", "col5"]
)

df["target"] = y

print(df.shape)
df.head()

## Function for row sampling

This function randomly selects a percentage of rows from the dataset.

`replace=True` means the same row can be selected more than once.  
This is called **bootstrap sampling**.


In [ ]:
# function for row sampling

def sample_rows(df, percent):
    return df.sample(
        int(percent * df.shape[0]),
        replace=True,
        random_state=random.randint(1, 1000)
    )

In [ ]:
df1 = sample_rows(df, 0.2)
df2 = sample_rows(df, 0.2)
df3 = sample_rows(df, 0.1)

df3.shape

In [ ]:
df1.head()

## Train three Decision Tree models using sampled rows

Each tree gets a different random sample of rows.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf1 = DecisionTreeClassifier(random_state=42)
clf2 = DecisionTreeClassifier(random_state=42)
clf3 = DecisionTreeClassifier(random_state=42)

In [ ]:
clf1.fit(df1.iloc[:, 0:5], df1.iloc[:, -1])
clf2.fit(df2.iloc[:, 0:5], df2.iloc[:, -1])
clf3.fit(df3.iloc[:, 0:5], df3.iloc[:, -1])

## Plot one tree

This visualizes one decision tree from the group of trees.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(18, 10))
plot_tree(
    clf1,
    feature_names=["col1", "col2", "col3", "col4", "col5"],
    class_names=["0", "1"],
    filled=True,
    rounded=True
)
plt.show()

## Predict using one trained tree

The input has 5 feature values because the model was trained using 5 columns.


In [ ]:
clf1.predict(
    np.array([0.51380, -1.76815, -1.80718, -1.57932, -0.03909]).reshape(1, 5)
)

# Feature Sampling

This function randomly selects a percentage of feature columns.

The target column is not included in feature sampling.


In [ ]:
# function for feature sampling

def sample_features(df, percent):
    cols = random.sample(
        df.columns.tolist()[:-1],
        int(percent * (df.shape[1] - 1))
    )
    return df[cols]

In [ ]:
sample_features(df, 0.6).head()

# Combined Sampling

This combines both ideas:

1. First select random rows
2. Then select random features from those rows


In [ ]:
# function for combined sampling

def combined_sampling(df, row_percent, col_percent):
    new_df = sample_rows(df, row_percent)
    return sample_features(new_df, col_percent)

In [ ]:
combined_sampling(df, 0.5, 0.6).head()

# Important Note

The screenshot showed examples of fitting decision trees after sampling.  
For practical training, the target column must be present during fitting.

The helper function below samples rows and selected features, then attaches the target column again.


In [ ]:
def combined_sampling_with_target(df, row_percent, col_percent):
    new_df = sample_rows(df, row_percent)

    cols = random.sample(
        new_df.columns.tolist()[:-1],
        int(col_percent * (new_df.shape[1] - 1))
    )

    cols.append("target")

    return new_df[cols]

In [ ]:
tree_df1 = combined_sampling_with_target(df, 0.5, 0.8)
tree_df2 = combined_sampling_with_target(df, 0.5, 0.8)
tree_df3 = combined_sampling_with_target(df, 0.5, 0.8)

tree_df1.head()

In [ ]:
clf1 = DecisionTreeClassifier(random_state=42)
clf2 = DecisionTreeClassifier(random_state=42)
clf3 = DecisionTreeClassifier(random_state=42)

clf1.fit(tree_df1.iloc[:, :-1], tree_df1.iloc[:, -1])
clf2.fit(tree_df2.iloc[:, :-1], tree_df2.iloc[:, -1])
clf3.fit(tree_df3.iloc[:, :-1], tree_df3.iloc[:, -1])

# Final Concept

Random Forest works by creating many Decision Trees.

Each tree is trained on:
- Different random rows
- Different random features

Then all trees vote together to make the final prediction.
